# K Folds Cross Validation


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_digits

In [4]:
digits = load_digits()
dir(digits)

['DESCR', 'data', 'feature_names', 'frame', 'images', 'target', 'target_names']

In [ ]:
df = pd.DataFrame(digits.data, columns=digits.feature_names)
df["target"] = digits.target
df["target_name"] = df.target.apply(lambda x: digits.target_names[x])
df.head()

,pixel_0_0,pixel_0_1,pixel_0_2,pixel_0_3,pixel_0_4,pixel_0_5,pixel_0_6,pixel_0_7,pixel_1_0,pixel_1_1,...,pixel_7_0,pixel_7_1,pixel_7_2,pixel_7_3,pixel_7_4,pixel_7_5,pixel_7_6,pixel_7_7,target,target_name
0,0.0,0.0,5.0,13.0,9.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,6.0,13.0,10.0,0.0,0.0,0.0,0,0
1,0.0,0.0,0.0,12.0,13.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,11.0,16.0,10.0,0.0,0.0,1,1
2,0.0,0.0,0.0,4.0,15.0,12.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,3.0,11.0,16.0,9.0,0.0,2,2
3,0.0,0.0,7.0,15.0,13.0,1.0,0.0,0.0,0.0,8.0,...,0.0,0.0,7.0,13.0,13.0,9.0,0.0,0.0,3,3
4,0.0,0.0,0.0,1.0,11.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,2.0,16.0,4.0,0.0,0.0,4,4


In [6]:
X = digits.data
y = digits.target

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [18]:
LinReg = LinearRegression()
LinReg.fit(X_train, y_train)
LinReg.score(X_test, y_test)

0.5756252064695238

In [19]:
LogReg = LogisticRegression(max_iter=2000)
LogReg.fit(X_train, y_train)
LogReg.score(X_test, y_test)

0.9666666666666667

In [20]:
svm = SVC()
svm.fit(X_train, y_train)
svm.score(X_test, y_test)

0.9925925925925926

In [21]:
rfc = RandomForestClassifier()
rfc.fit(X_train, y_train)
rfc.score(X_test, y_test)

0.975925925925926

## Why K-folds comes into picture?

- We can train a model in 3 ways :
  1. We can use the whole dataset, but then again when we will use the same dataset to test that model then it may be biased.
  2. We can use train_test_split method to split the whole dataset into to portion for train and test (generally 80% train and 20% test) and then we can overcome the problem of biasness, but as train_test_split method splits data in a random manner so data for all different class may not be present equivalently which reduces accuracy of model.
  3. So to overcome these above problems we can use K-folds.We can divide the data into K folds or K parts and then iteratively train and test the model with all different folds and then the average score of model will be more accurate.


In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5)
for train, test in kf.split([1, 2, 3, 4, 5, 6, 7, 8, 9]):
    print(train, test)

[2 3 4 5 6 7 8] [0 1]
[0 1 4 5 6 7 8] [2 3]
[0 1 2 3 6 7 8] [4 5]
[0 1 2 3 4 5 8] [6 7]
[0 1 2 3 4 5 6 7] [8]


In [36]:
def model_score(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    return model.score(X_test, y_test)

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5)
logistic = []
svm = []
rf = []
for train_index, test_index in skf.split(digits.data, digits.target):
    X_train, X_test, y_train, y_test = (
        digits.data[train_index],
        digits.data[test_index],
        digits.target[train_index],
        digits.target[test_index],
    )
    logistic.append(
        model_score(
            LogisticRegression(max_iter=1500, C=0.5), X_train, X_test, y_train, y_test
        )
    )
    svm.append(model_score(SVC(C=2), X_train, X_test, y_train, y_test))
    rf.append(
        model_score(
            RandomForestClassifier(n_estimators=180), X_train, X_test, y_train, y_test
        )
    )

In [80]:
print(logistic)
print(svm)
print(rf)

[0.9222222222222223, 0.8722222222222222, 0.947075208913649, 0.9415041782729805, 0.8969359331476323]
[0.9805555555555555, 0.9472222222222222, 0.9832869080779945, 0.9888579387186629, 0.9526462395543176]
[0.9333333333333333, 0.9111111111111111, 0.9637883008356546, 0.9665738161559888, 0.924791086350975]


In [ ]:
# All the code above this cell is to understand the backend logic of K-fold cross validation
# Now just import the library and use that and enjoy!!

from sklearn.model_selection import cross_val_score

cross_val_score(
    LogisticRegression(max_iter=1500, solver="liblinear"), digits.data, digits.target
)

array([0.92222222, 0.88333333, 0.95264624, 0.95821727, 0.89415042])

In [86]:
cross_val_score(SVC(), digits.data, digits.target)

array([0.96111111, 0.94444444, 0.98328691, 0.98885794, 0.93871866])

In [ ]:
cross_val_score(
    RandomForestClassifier(n_estimators=180), digits.data, digits.target
).mean()

np.float64(0.9410290931600123)

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data
y = iris.target

In [113]:
cross_val_score(LogisticRegression(max_iter=len(X)), X, y, cv=10).mean()

np.float64(0.9733333333333334)

In [150]:
cross_val_score(SVC(C=10, kernel='rbf'), X, y, cv=10, verbose=10)

[CV] START .....................................................................
[CV] END ................................ score: (test=1.000) total time=   0.0s
[CV] START .....................................................................
[CV] END ................................ score: (test=0.933) total time=   0.0s
[CV] START .....................................................................
[CV] END ................................ score: (test=1.000) total time=   0.0s
[CV] START .....................................................................
[CV] END ................................ score: (test=1.000) total time=   0.0s
[CV] START .....................................................................
[CV] END ................................ score: (test=0.933) total time=   0.0s
[CV] START .....................................................................
[CV] END ................................ score: (test=1.000) total time=   0.0s
[CV] START .................

[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   4 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done   7 tasks      | elapsed:    0.0s


array([1.        , 0.93333333, 1.        , 1.        , 0.93333333,
       1.        , 0.93333333, 1.        , 1.        , 1.        ])

In [151]:
cross_val_score(RandomForestClassifier(n_estimators=300), X, y, cv=10,  verbose=12).mean()

[CV] START .....................................................................
[CV] END ................................ score: (test=1.000) total time=   0.4s
[CV] START .....................................................................


[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.4s


[CV] END ................................ score: (test=0.933) total time=   0.4s
[CV] START .....................................................................


[Parallel(n_jobs=1)]: Done   2 tasks      | elapsed:    0.9s


[CV] END ................................ score: (test=1.000) total time=   0.4s
[CV] START .....................................................................


...

[CV] END ................................ score: (test=1.000) total time=   0.4s
[CV] START .....................................................................


[Parallel(n_jobs=1)]: Done   9 tasks      | elapsed:    4.3s


[CV] END ................................ score: (test=1.000) total time=   0.4s


[Parallel(n_jobs=1)]: Done  10 tasks      | elapsed:    4.8s
[Parallel(n_jobs=1)]: Done  10 tasks      | elapsed:    4.8s


np.float64(0.9666666666666666)